In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

In [ ]:
df = pd.read_csv("data/EDA.csv")

df.head()


In [ ]:
#Description:
#AHT represents the average handling time (seconds per answered case) for each employee during a working day.
#AHT = Total On Queue Time / Total Answered Case

In [ ]:
#1. Overall Performance Overview
#Number of handled tickets and AHT on average:
df[["Answered", "AHT"]].describe()



In [ ]:
df[["Answered", "AHT"]].median()

In [ ]:
Head_count = df.groupby(["Shift","Month"]).agg(
    employee = ("WD EID", "nunique")
)
Head_count.reset_index()



In [ ]:
shift_check = df.groupby(["WD EID", "Emp Name"])["Shift"].agg(
    shift_count = (lambda x: ", ".join(set(x))),
    count = ("nunique")
).reset_index()

shift_check

In [ ]:
#Employees who have had to change their working schedule more than once
shift_check[shift_check["count"] != 1]

,WD EID,Emp Name,shift_count,count
0,102496670,Dieu Tien,"Noon, Day",2
1,102496681,Nguyen Thi Phuong Thuy Nguyen,"Noon, Night",2
2,102845218,Nguyen Ngoc Anh Thu,"Noon, Day",2
3,102893381,Vu Van Khang,"Noon, Night",2
4,103193904,Vuong Hoang Bao,"Noon, Day, Night",3
5,103222889,Le Hau,"Day, Night",2
6,103286054,Nguyen Y Bao Han,"Day, Night",2
7,103293621,Nguyen Tran Anh Tu,"Noon, Night",2
8,103304193,Truong Thi Thanh Tam,"Day, Night",2
9,103309812,Tran Hoang Anh,"Noon, Day",2


In [117]:
#Employees who can keep their work schedule stay still 
shift_check[shift_check["count"] == 1]

,WD EID,Emp Name,shift_count,count
10,103310980,Nguyen Thuy Duong,Night,1
12,103324502,Tran Nguyen Hoang Trieu,Noon,1
13,103338101,Le Phi Hoang,Noon,1
14,103338297,Cao Minh Hoang,Noon,1
15,103345473,Nguyen Duc Thuan,Noon,1
16,103431523,Tran Thanh Viet Ha,Noon,1
18,103455202,Huynh Le Thuy Thanh,Noon,1
21,103456015,Nguyen Ngoc Thanh Tam,Night,1
22,103471626,Nguyen Ho Minh Dai,Night,1


In [ ]:
#Use histogram to explore the variance of AHT and Answered number

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df["AHT"].plot.hist(
    bins=40,
    ax=axes[0],
    color="cyan",
    edgecolor="white",
    alpha=0.8
)
axes[0].set_title("Overall Distribution of AHT")

df["Answered"].plot.hist(
    bins=20,
    ax=axes[1],
    color="navy",
    edgecolor="white",
    alpha=0.8
)
axes[1].set_title("Overall Distribution of Answered")

plt.tight_layout()


In [ ]:
stats.probplot(df["AHT"], dist="norm", plot=plt)

plt.show()

In [ ]:
stats.probplot(df["Answered"], dist="norm", plot=plt)

plt.show()

In [ ]:
q1_aht, median_aht, q3_aht = np.percentile(df.AHT, [25, 50, 75])
iqr_aht = q3_aht - q1_aht

q1_ans, median_ans, q3_ans = np.percentile(df.Answered, [25, 50, 75])
iqr_ans = q3_ans- q1_ans

aht = [q1_aht, q3_aht, median_aht, iqr_aht]
ans = [q1_ans, q3_ans, median_ans, iqr_ans]

attribute = ["q1", "q3", "median", "IQR"]

pd.DataFrame({
    "attribute": attribute,
    "AHT_box": aht,
    "Answered_box": ans  
})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(
    data=df,
    x="Month",
    y="AHT",
    ax=axes[0]
)

sns.boxplot(
    data=df,
    x="Month",
    y="Answered",
    ax=axes[1]
)

plt.tight_layout()

**Descriptive Statistics**

The median AHT is 411 seconds per answered case, while the median number of answered cases is 58. The noticeable gap between the minimum and maximum values suggests substantial variation in employee performance across workdays.

**Histogram**

The histogram indicates that AHT is positively skewed. Most observations fall between approximately 360 and 510 seconds, while a relatively small number of observations extend into the upper tail. These high values may correspond to workdays involving more complex customer cases or operational disruptions.

**QQ Plot**

The QQ plot further confirms that AHT deviates from a normal distribution. While observations near the center align reasonably well with the theoretical normal line, the upper tail bends upward, indicating a heavy right tail and several unusually high AHT observations.

In [ ]:
#2. Time Series Analysis

sns.pointplot(
    data=df,
    x="Shift",
    y="AHT",
    estimator=np.mean,
    errorbar=("ci", 95),
    capsize=0.15
)



In [ ]:
df_shift_insight = df.groupby("Shift").agg(
    avg_AHT=("AHT", "mean"),
    avg_Answered=("Answered", "mean")
).reset_index()

overall_mean_aht = df["AHT"].mean()
overall_mean_answered = df["Answered"].mean()


df_shift_insight["diff_AHT_vs_overall"] = df_shift_insight["avg_AHT"] - overall_mean_aht
df_shift_insight["diff_Answered_vs_overall"] = df_shift_insight["avg_Answered"] - overall_mean_answered

df_shift_insight

#Description:
#diff_AHT_vs_overall < 0 is better
#diff_Answered_vs_overall > 0 is better



In [ ]:
df_AHT_Ans_trend = df.groupby(["Month", "Dayofweek"]).agg(
    avg_AHT_weekday = ("AHT", "mean"),
    avg_Answered_weekday = ("Answered", "mean")
).reset_index()
Month_AHT_Ans = df.groupby("Month").agg(
    avg_AHT_month = ("AHT", "mean"),
    avg_Answered_month = ("Answered", "mean")
)

reports = pd.merge(df_AHT_Ans_trend, Month_AHT_Ans, how="inner", left_on="Month", right_on="Month")

reports = reports.assign(
    AHT_diff = (((reports.avg_AHT_weekday - reports.avg_AHT_month)/reports.avg_AHT_month)*100).map("{:.2f}%".format),
    Answered_diff = (((reports.avg_Answered_weekday - reports.avg_Answered_month)/reports.avg_AHT_weekday)*100).map("{:.2f}%".format),
)

reports


Part 2 actually brings on many interesting insights covering critial parts of our dataset.

To be in the first place, Day, Noon and Night shifts average handling time are almost the same, skeptically, Noon shift has the highest AHT but it does not look like a big deal here. The same instance shared with handled cases number diff.
However, by somehow, Night shift which has lowest AHT is also the one with lowest handled cases.



In [ ]:
#AHT
sns.catplot(data = df, x = "Dayofweek", y = "AHT", 
               col = "Shift",
               row= "Month",
               hue="Shift",
               kind= "point",
               palette = "plasma",
               estimator=np.median,
               errorbar=("ci", 95)
)
#Answered
sns.catplot(data = df, x = "Dayofweek", y = "Answered", 
               col = "Shift",
               row= "Month",
               hue="Shift",
               kind= "point",
               palette = "plasma",
               estimator=np.median,
               errorbar=("ci", 95)
)

#Description: Week starts at 0 meaning Monday and ends at 6 meaning Sunday